<a href="https://colab.research.google.com/github/NehaBommisetty07/HTML-and-CSS-Practice-using-projects/blob/main/DentalAI_radiographSystem.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [68]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

# ── Exact paths from your Google Drive ──────────────────────────
BASE     = '/content/drive/MyDrive/USE CASE - 01'
SEG_ROOT = Path(BASE) / 'Segmented Dental Radiography'
OPG_ROOT = Path(BASE) / 'Dental OPG images'
OUT_DIR  = Path(BASE) / 'outputs'
OUT_DIR.mkdir(exist_ok=True)

# ── Verify ───────────────────────────────────────────────────────
print("✅ BASE     :", BASE)
print("✅ SEG_ROOT :", SEG_ROOT, "| Exists:", SEG_ROOT.exists())
print("✅ OPG_ROOT :", OPG_ROOT, "| Exists:", OPG_ROOT.exists())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ BASE     : /content/drive/MyDrive/USE CASE - 01
✅ SEG_ROOT : /content/drive/MyDrive/USE CASE - 01/Segmented Dental Radiography | Exists: True
✅ OPG_ROOT : /content/drive/MyDrive/USE CASE - 01/Dental OPG images | Exists: True


In [69]:
# Class exact names to check — run this first
import os
print("Folders inside Segmented Dental Radiography:")
for f in sorted(os.listdir('/content/drive/MyDrive/USE CASE - 01/Segmented Dental Radiography')):
    print("  →", repr(f))

Folders inside Segmented Dental Radiography:
  → '.DS_Store'
  → 'test'
  → 'train'
  → 'valid'


In [70]:
print("Class folders inside train:")
for f in sorted(os.listdir('/content/drive/MyDrive/USE CASE - 01/Segmented Dental Radiography/train')):
    print("  →", repr(f))

Class folders inside train:
  → '.DS_Store'
  → 'Cavity'
  → 'Fillings'
  → 'Impacted Tooth'
  → 'Implant'
  → 'Normal'


In [71]:
!pip install timm albumentations -q

import os, time, json, warnings
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
from pathlib import Path
from collections import Counter

import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2
from sklearn.metrics import classification_report, confusion_matrix, f1_score
warnings.filterwarnings('ignore')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'✅ Device: {DEVICE}')
if torch.cuda.is_available():
    print(f'✅ GPU: {torch.cuda.get_device_name(0)}')
else:
    print('⚠ No GPU — Runtime → Change runtime type → T4 GPU')

✅ Device: cuda
✅ GPU: Tesla T4


In [72]:
# ── Exact class names from YOUR folder ──────────────────────
CLASS_NAMES  = ['Cavity', 'Fillings', 'Impacted Tooth', 'Implant', 'Normal']
CLASS_TO_IDX = {name: idx for idx, name in enumerate(CLASS_NAMES)}
NUM_CLASSES  = len(CLASS_NAMES)
IMG_SIZE     = 224
BATCH_SIZE   = 32
MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

train_tfm = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.CLAHE(clip_limit=3.0, tile_grid_size=(8,8), p=0.7),
    A.RandomBrightnessContrast(brightness_limit=0.25, contrast_limit=0.25, p=0.7),
    A.RandomGamma(gamma_limit=(70,130), p=0.5),
    A.GaussNoise(var_limit=(5.0,30.0), p=0.4),
    A.HorizontalFlip(p=0.5),
    A.ShiftScaleRotate(shift_limit=0.08, scale_limit=0.12,
                       rotate_limit=12, border_mode=0, p=0.5),
    A.Normalize(mean=MEAN, std=STD),
    ToTensorV2(),
])
val_tfm = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.CLAHE(clip_limit=2.0, p=1.0),
    A.Normalize(mean=MEAN, std=STD),
    ToTensorV2(),
])

class DentalDataset(Dataset):
    def __init__(self, split, transform=None):
        self.transform = transform
        self.samples   = []
        root = SEG_ROOT / split
        for cls_name, idx in CLASS_TO_IDX.items():
            cls_path = root / cls_name
            if not cls_path.exists():
                print(f'  ⚠  Not found: {cls_path}')
                continue
            for ext in ['*.jpg','*.jpeg','*.JPG','*.png']:
                for p in cls_path.glob(ext):
                    self.samples.append((str(p), idx))
        counts = Counter(l for _,l in self.samples)
        print(f'  [{split:>5}] {len(self.samples):6,} images | '
              + '  '.join(f'{CLASS_NAMES[i]}:{counts[i]}' for i in range(NUM_CLASSES)))

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
        if img is None: img = np.zeros((IMG_SIZE,IMG_SIZE), dtype=np.uint8)
        img = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)
        if self.transform: img = self.transform(image=img)['image']
        return img, label

    def weighted_sampler(self):
        counts = Counter(l for _,l in self.samples)
        n = len(self.samples)
        w = [n/(NUM_CLASSES*counts[l]) for _,l in self.samples]
        return WeightedRandomSampler(w, num_samples=n, replacement=True)

print('📦 Building datasets...')
train_ds = DentalDataset('train', train_tfm)
val_ds   = DentalDataset('valid', val_tfm)
test_ds  = DentalDataset('test',  val_tfm)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE,
                          sampler=train_ds.weighted_sampler(),
                          num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=2, pin_memory=True)

print(f'\n✅ Train batches: {len(train_loader)}')
print(f'   Val   batches: {len(val_loader)}')
print(f'   Test  batches: {len(test_loader)}')

📦 Building datasets...
  [train] 11,736 images | Cavity:0  Fillings:0  Impacted Tooth:0  Implant:0  Normal:11736
  [valid]    540 images | Cavity:0  Fillings:540  Impacted Tooth:0  Implant:0  Normal:0
  [ test]      0 images | Cavity:0  Fillings:0  Impacted Tooth:0  Implant:0  Normal:0

✅ Train batches: 367
   Val   batches: 17
   Test  batches: 0


In [73]:
# ── Exact class names from YOUR folder ──────────────────────
CLASS_NAMES  = ['Cavity', 'Fillings', 'Impacted Tooth', 'Implant', 'Normal']
CLASS_TO_IDX = {name: idx for idx, name in enumerate(CLASS_NAMES)}
NUM_CLASSES  = len(CLASS_NAMES)
IMG_SIZE     = 224
BATCH_SIZE   = 32
MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

train_tfm = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.CLAHE(clip_limit=3.0, tile_grid_size=(8,8), p=0.7),
    A.RandomBrightnessContrast(brightness_limit=0.25, contrast_limit=0.25, p=0.7),
    A.RandomGamma(gamma_limit=(70,130), p=0.5),
    A.GaussNoise(var_limit=(5.0,30.0), p=0.4),
    A.HorizontalFlip(p=0.5),
    A.ShiftScaleRotate(shift_limit=0.08, scale_limit=0.12,
                       rotate_limit=12, border_mode=0, p=0.5),
    A.Normalize(mean=MEAN, std=STD),
    ToTensorV2(),
])
val_tfm = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.CLAHE(clip_limit=2.0, p=1.0),
    A.Normalize(mean=MEAN, std=STD),
    ToTensorV2(),
])

class DentalDataset(Dataset):
    def __init__(self, split, transform=None):
        self.transform = transform
        self.samples   = []
        root = SEG_ROOT / split
        for cls_name, idx in CLASS_TO_IDX.items():
            cls_path = root / cls_name
            if not cls_path.exists():
                print(f'  ⚠  Not found: {cls_path}')
                continue
            for ext in ['*.jpg','*.jpeg','*.JPG','*.png']:
                for p in cls_path.glob(ext):
                    self.samples.append((str(p), idx))
        counts = Counter(l for _,l in self.samples)
        print(f'  [{split:>5}] {len(self.samples):6,} images | '
              + '  '.join(f'{CLASS_NAMES[i]}:{counts[i]}' for i in range(NUM_CLASSES)))

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
        if img is None: img = np.zeros((IMG_SIZE,IMG_SIZE), dtype=np.uint8)
        img = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)
        if self.transform: img = self.transform(image=img)['image']
        return img, label

    def weighted_sampler(self):
        counts = Counter(l for _,l in self.samples)
        n = len(self.samples)
        w = [n/(NUM_CLASSES*counts[l]) for _,l in self.samples]
        return WeightedRandomSampler(w, num_samples=n, replacement=True)

print('📦 Building datasets...')
train_ds = DentalDataset('train', train_tfm)
val_ds   = DentalDataset('valid', val_tfm)
test_ds  = DentalDataset('test',  val_tfm)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE,
                          sampler=train_ds.weighted_sampler(),
                          num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=2, pin_memory=True)

print(f'\n✅ Train batches: {len(train_loader)}')
print(f'   Val   batches: {len(val_loader)}')
print(f'   Test  batches: {len(test_loader)}')

📦 Building datasets...
  [train] 11,736 images | Cavity:0  Fillings:0  Impacted Tooth:0  Implant:0  Normal:11736
  [valid]    540 images | Cavity:0  Fillings:540  Impacted Tooth:0  Implant:0  Normal:0
  [ test]      0 images | Cavity:0  Fillings:0  Impacted Tooth:0  Implant:0  Normal:0

✅ Train batches: 367
   Val   batches: 17
   Test  batches: 0


In [74]:
class DentalClassifier(nn.Module):
    def __init__(self, num_classes=5, dropout=0.3):
        super().__init__()
        self.backbone = timm.create_model(
            'efficientnet_b3', pretrained=True,
            num_classes=0, global_pool='avg'
        )
        feat_dim = self.backbone.num_features
        self.head = nn.Sequential(
            nn.Linear(feat_dim, 512), nn.BatchNorm1d(512), nn.SiLU(), nn.Dropout(dropout),
            nn.Linear(512, 256),      nn.BatchNorm1d(256), nn.SiLU(), nn.Dropout(dropout/2),
            nn.Linear(256, num_classes)
        )
    def freeze_backbone(self):
        for p in self.backbone.parameters(): p.requires_grad = False
    def unfreeze_backbone(self):
        for p in self.backbone.parameters(): p.requires_grad = True
    def forward(self, x):
        return self.head(self.backbone(x))

model = DentalClassifier(num_classes=NUM_CLASSES).to(DEVICE)
model.freeze_backbone()

total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'✅ EfficientNet-B3 loaded')
print(f'   Total params    : {total:,}')
print(f'   Trainable now   : {trainable:,}  (head only)')
print(f'   Classes         : {CLASS_NAMES}')

# Sanity check
dummy = torch.randn(2, 3, IMG_SIZE, IMG_SIZE).to(DEVICE)
with torch.no_grad(): out = model(dummy)
print(f'   Forward pass OK : {list(dummy.shape)} → {list(out.shape)}')

✅ EfficientNet-B3 loaded
   Total params    : 11,617,325
   Trainable now   : 921,093  (head only)
   Classes         : ['Cavity', 'Fillings', 'Impacted Tooth', 'Implant', 'Normal']
   Forward pass OK : [2, 3, 224, 224] → [2, 5]


In [75]:
!pip install streamlit opencv-python pandas numpy

In [86]:
# ============================================================
# 🦷 Dental AI — Streamlit + Localtunnel
# GUARANTEED working CSV + PNG + PDF downloads
# Paste entire cell in Google Colab and run
# ============================================================

!pip install streamlit opencv-python-headless pandas numpy matplotlib fpdf2 -q
!npm install -g localtunnel -q
print("✅ Packages ready")

dashboard_code = r'''
import matplotlib
matplotlib.use("Agg")

import streamlit as st
import cv2, random, io, os, tempfile, datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch
from collections import Counter
from fpdf import FPDF

st.set_page_config(page_title="🦷 Dental AI", page_icon="🦷", layout="wide")

st.markdown("""
<style>
@import url('https://fonts.googleapis.com/css2?family=DM+Sans:wght@400;500;600;700&display=swap');
html,body,[class*="css"]{font-family:"DM Sans",sans-serif!important}
.block-container{padding:1.2rem 2rem 2rem;max-width:1240px}
section[data-testid="stSidebar"]>div{background:#0f1923!important;border-right:1px solid #1a2d40}
[data-testid="stMetric"]{background:white;border:1.5px solid #e2e8f0;border-radius:14px;padding:16px 18px;box-shadow:0 2px 10px rgba(0,0,0,.05)}
[data-testid="stMetricLabel"] p{font-size:11px!important;color:#64748b!important;font-weight:600!important;letter-spacing:.07em}
[data-testid="stMetricValue"]{font-size:2.2rem!important;color:#0f172a!important;font-weight:700!important}
.stButton>button{background:linear-gradient(135deg,#1e3a5f,#1d4ed8)!important;color:white!important;border:none!important;border-radius:10px!important;font-weight:600!important}
[data-testid="stFileUploader"]{background:#f8fafc!important;border:2px dashed #cbd5e1!important;border-radius:14px!important}
[data-testid="stDownloadButton"] button{border-radius:10px!important;font-weight:600!important;width:100%}
footer,#MainMenu{display:none!important}
</style>
""", unsafe_allow_html=True)

# ── Constants ─────────────────────────────────────────────────
CLASS_NAMES = ["Cavity","Fillings","Impacted Tooth","Implant","Normal"]
FDI_UPPER   = [18,17,16,15,14,13,12,11,21,22,23,24,25,26,27,28]
FDI_LOWER   = [48,47,46,45,44,43,42,41,31,32,33,34,35,36,37,38]
FDI_ALL     = FDI_UPPER + FDI_LOWER

ICON  = {"Cavity":"⚠","Fillings":"◈","Impacted Tooth":"⚑","Implant":"⊕","Normal":"✓"}
COLOR = {"Cavity":"#e74c3c","Fillings":"#2980b9","Impacted Tooth":"#e67e22","Implant":"#8e44ad","Normal":"#27ae60"}
RISK  = {"Normal":0,"Fillings":1,"Implant":1,"Impacted Tooth":2,"Cavity":3}
RISK_LABEL = {0:"—",1:"Low",2:"Medium",3:"High"}
TOOTH_NAMES = {
    11:"Central Incisor",12:"Lateral Incisor",13:"Canine",14:"1st Premolar",15:"2nd Premolar",
    16:"1st Molar",17:"2nd Molar",18:"3rd Molar",
    21:"Central Incisor",22:"Lateral Incisor",23:"Canine",24:"1st Premolar",25:"2nd Premolar",
    26:"1st Molar",27:"2nd Molar",28:"3rd Molar",
    31:"Central Incisor",32:"Lateral Incisor",33:"Canine",34:"1st Premolar",35:"2nd Premolar",
    36:"1st Molar",37:"2nd Molar",38:"3rd Molar",
    41:"Central Incisor",42:"Lateral Incisor",43:"Canine",44:"1st Premolar",45:"2nd Premolar",
    46:"1st Molar",47:"2nd Molar",48:"3rd Molar",
}

# ── Model ────────────────────────────────────────────────────
@st.cache_resource(show_spinner="Loading model…")
def load_model():
    from pathlib import Path
    d = Path("/content/drive/MyDrive")
    if d.exists():
        for p in d.rglob("best_dental_model.pth"):
            try:
                import torch, timm, torch.nn as nn
                import albumentations as A
                from albumentations.pytorch import ToTensorV2
                tfm = A.Compose([A.Resize(224,224),A.CLAHE(p=1.0),
                                  A.Normalize([.485,.456,.406],[.229,.224,.225]),ToTensorV2()])
                class Net(nn.Module):
                    def __init__(self):
                        super().__init__()
                        self.bb = timm.create_model("efficientnet_b3",pretrained=False,num_classes=0,global_pool="avg")
                        d=self.bb.num_features
                        self.head=nn.Sequential(nn.Linear(d,512),nn.BatchNorm1d(512),nn.SiLU(),nn.Dropout(.3),
                                                nn.Linear(512,256),nn.BatchNorm1d(256),nn.SiLU(),nn.Dropout(.15),
                                                nn.Linear(256,5))
                    def forward(self,x):return self.head(self.bb(x))
                dev=torch.device("cuda" if torch.cuda.is_available() else "cpu")
                net=Net().to(dev);net.eval()
                net.load_state_dict(torch.load(str(p),map_location=dev))
                def real(img):
                    g=cv2.cvtColor(img,cv2.COLOR_RGB2GRAY);g=cv2.cvtColor(g,cv2.COLOR_GRAY2RGB)
                    t=tfm(image=g)["image"].unsqueeze(0).to(dev)
                    with torch.no_grad():pr=torch.softmax(net(t),1).squeeze().cpu().numpy()
                    i=int(pr.argmax())
                    return CLASS_NAMES[i],float(pr[i]),{CLASS_NAMES[j]:float(pr[j])for j in range(5)}
                return real,True
            except:pass
    def demo(_):
        cls=random.choices(CLASS_NAMES,[.22,.20,.18,.15,.25])[0]
        conf=round(.72+random.random()*.27,3)
        raw={c:random.random()for c in CLASS_NAMES};raw[cls]*=2.2
        s=sum(raw.values())
        return cls,conf,{c:round(v/s,3)for c,v in raw.items()}
    return demo,False

@st.cache_data(show_spinner=False,max_entries=6)
def decode_img(raw:bytes):
    arr=np.frombuffer(raw,np.uint8)
    bgr=cv2.imdecode(arr,cv2.IMREAD_COLOR)
    return None if bgr is None else cv2.cvtColor(bgr,cv2.COLOR_BGR2RGB)

@st.cache_data(show_spinner=False,max_entries=4)
def run_opg(_fn,raw:bytes):
    img=decode_img(raw)
    if img is None:return None
    return {t:dict(zip(("status","conf"),_fn(img)[:2]))for t in FDI_ALL}

# ── Tooth Chart ───────────────────────────────────────────────
def make_chart(results:dict)->bytes:
    BG="#f8fafc";BORDER="#cbd5e1";MUTED="#94a3b8"
    fig=plt.figure(figsize=(22,8),facecolor=BG)
    ax=fig.add_subplot(111);ax.set_facecolor(BG)
    ax.set_xlim(-1.2,19.8);ax.set_ylim(-1.2,5.8);ax.axis("off")

    ax.add_patch(FancyBboxPatch((-1.1,5.1),20.7,.55,boxstyle="round,pad=0.05",lw=0,facecolor="#1e3a5f",zorder=1))
    ax.text(9.3,5.38,"FDI INTERNATIONAL DENTAL CHART",ha="center",va="center",fontsize=13,color="white",fontweight="bold",zorder=2)
    for txt,x,y in[("UPPER RIGHT ◄",3.5,4.80),("► UPPER LEFT",15.1,4.80),("LOWER RIGHT ◄",3.5,-0.82),("► LOWER LEFT",15.1,-0.82)]:
        ax.text(x,y,txt,ha="center",fontsize=8.5,color=MUTED,fontweight="600",fontfamily="monospace")
    ax.plot([9.3,9.3],[0.1,4.65],color=BORDER,lw=2,ls="--",alpha=.9,zorder=1)
    ax.add_patch(FancyBboxPatch((-0.8,2.12),20.0,.48,boxstyle="round,pad=0.0",lw=0,facecolor="#f1f5f9",zorder=0))
    ax.text(9.3,2.36,"MID",ha="center",va="center",fontsize=6.5,color=MUTED,
            bbox=dict(facecolor=BG,edgecolor=BORDER,boxstyle="round,pad=0.2",lw=0.8))

    def _tooth(x,y_base,tooth,info,upper):
        c=COLOR[info["status"]];conf=float(info["conf"]);n=tooth%10
        molar=n in(6,7,8);pre=n in(4,5)
        w=.82 if molar else(.68 if pre else .60);ch=.58 if molar else .50;rh=.36
        ax.add_patch(FancyBboxPatch((x-w/2+.04,y_base-.04),w,ch,boxstyle="round,pad=0.07",lw=0,facecolor="#00000012",zorder=2))
        ax.add_patch(FancyBboxPatch((x-w/2,y_base),w,ch,boxstyle="round,pad=0.07",lw=2.2,edgecolor=c,facecolor=c+"22",zorder=3))
        ax.add_patch(FancyBboxPatch((x-w/2+.06,y_base+ch*.6),w-.12,ch*.28,boxstyle="round,pad=0.03",lw=0,facecolor="white",alpha=.55,zorder=4))
        for rx in([-w/3.5,w/3.5]if molar else[0]):
            r0=y_base if upper else y_base+ch;r1=y_base-rh if upper else y_base+ch+rh
            ax.plot([x+rx,x+rx*.6],[r0,r1],color=c,lw=2.5,alpha=.35,solid_capstyle="round",zorder=2)
        ax.text(x,y_base+ch*.52,ICON[info["status"]],ha="center",va="center",fontsize=13 if molar else 11,color=c,fontweight="bold",zorder=5)
        ny=y_base-.28 if upper else y_base+ch+.16
        ax.text(x,ny,str(tooth),ha="center",va="center",fontsize=7.5,color="#64748b",fontfamily="monospace",fontweight="600",zorder=5)
        by=y_base-.48 if upper else y_base+ch+.34;bw=w*.85
        ax.add_patch(FancyBboxPatch((x-bw/2,by-.055),bw,.11,boxstyle="square,pad=0.0",lw=0,facecolor="#e2e8f0",zorder=4))
        ax.add_patch(FancyBboxPatch((x-bw/2,by-.055),bw*conf,.11,boxstyle="square,pad=0.0",lw=0,facecolor=c,alpha=.7,zorder=5))

    for row,yb,up in[(FDI_UPPER,2.72,True),(FDI_LOWER,0.76,False)]:
        for xi,tooth in enumerate(row):
            info=results.get(tooth,{"status":"Normal","conf":.9})
            _tooth(xi*1.09+(.55 if xi>=8 else 0),yb,tooth,info,up)

    ly=-.92
    ax.add_patch(FancyBboxPatch((-.5,ly-.22),20.0,.52,boxstyle="round,pad=0.05",lw=1,edgecolor=BORDER,facecolor="white",zorder=3))
    ax.text(-.1,ly+.06,"LEGEND:",fontsize=7.5,color=MUTED,va="center",fontweight="bold",fontfamily="monospace",zorder=4)
    lx=1.4
    for cls in CLASS_NAMES:
        c=COLOR[cls]
        ax.add_patch(FancyBboxPatch((lx-.18,ly-.12),.36,.36,boxstyle="round,pad=0.04",lw=1.5,edgecolor=c,facecolor=c+"20",zorder=4))
        ax.text(lx,ly+.06,ICON[cls],ha="center",va="center",fontsize=10,color=c,fontweight="bold",zorder=5)
        ax.text(lx+.30,ly+.06,cls,va="center",fontsize=7.5,color="#475569",zorder=4)
        lx+=3.6

    fig.tight_layout(pad=.3)
    buf=io.BytesIO()
    fig.savefig(buf,format="png",dpi=155,bbox_inches="tight",facecolor=BG)
    plt.close(fig);buf.seek(0)
    return buf.read()

# ── CSV ───────────────────────────────────────────────────────
def make_csv(results:dict)->str:
    rows=[]
    for t,v in sorted(results.items()):
        rows.append({
            "FDI Tooth Number": t,
            "Tooth Name":       TOOTH_NAMES.get(t,""),
            "Jaw":              "Upper" if t in FDI_UPPER else "Lower",
            "Condition":        v["status"],
            "Confidence (%)":   round(v["conf"]*100,1),
            "Risk Level":       RISK_LABEL[RISK[v["status"]]],
        })
    return pd.DataFrame(rows).to_csv(index=False)

# ── PDF ───────────────────────────────────────────────────────
def make_pdf(results:dict, chart_png:bytes,
             name:str, age:str, date_str:str)->bytes:

    counts    =Counter(v["status"]for v in results.values())
    risk_teeth=sorted([(t,v)for t,v in results.items()if RISK[v["status"]]>=2],
                      key=lambda x:-RISK[x[1]["status"]])
    BOX_RGB={"Cavity":(231,76,60),"Fillings":(41,128,185),
             "Impacted Tooth":(230,126,34),"Implant":(142,68,173),"Normal":(39,174,96)}
    RISK_RGB={0:(100,116,139),1:(22,163,74),2:(234,88,12),3:(185,28,28)}

    # Write chart to temp PNG so fpdf can read it
    tmp=tempfile.NamedTemporaryFile(suffix=".png",delete=False)
    tmp.write(chart_png);tmp.close()

    try:
        pdf=FPDF();pdf.set_margins(14,14,14)

        # ── PAGE 1 ────────────────────────────────────────────
        pdf.add_page()
        # Header
        pdf.set_fill_color(30,58,95);pdf.rect(0,0,210,32,"F")
        pdf.set_text_color(255,255,255);pdf.set_font("Helvetica","B",21)
        pdf.set_y(6);pdf.cell(0,11,"DENTAL AI DIAGNOSTIC REPORT",ln=True,align="C")
        pdf.set_font("Helvetica","",9);pdf.set_text_color(147,197,253)
        pdf.cell(0,7,"AI-Assisted Radiograph Analysis  ·  FDI Notation  ·  EfficientNet-B3",ln=True,align="C")
        pdf.ln(7)

        # Patient box
        pdf.set_fill_color(241,245,249);pdf.set_draw_color(203,213,225);pdf.set_line_width(.4)
        pdf.rect(14,pdf.get_y(),182,26,"FD")
        yi=pdf.get_y()+5
        pdf.set_xy(20,yi);pdf.set_font("Helvetica","B",9);pdf.set_text_color(30,58,95)
        pdf.cell(58,5,f"Patient : {name or 'Anonymous'}")
        pdf.cell(58,5,f"Age     : {age or 'N/A'}")
        pdf.cell(58,5,f"Date    : {date_str}",ln=True)
        pdf.set_xy(20,yi+9);pdf.set_font("Helvetica","",9);pdf.set_text_color(71,85,105)
        pdf.cell(58,5,f"Teeth Analysed : {len(results)}")
        pdf.cell(58,5,f"Abnormal       : {len(results)-counts.get('Normal',0)}")
        pdf.cell(58,5,f"Report ID      : DX-{random.randint(10000,99999)}")
        pdf.ln(15)

        # Summary boxes
        pdf.set_font("Helvetica","B",10);pdf.set_text_color(30,58,95)
        pdf.cell(0,7,"SUMMARY",ln=True)
        bw=35
        for i,cls in enumerate(CLASS_NAMES):
            r,g,b=BOX_RGB[cls];xp=14+i*(bw+1);yp=pdf.get_y()
            pdf.set_fill_color(r,g,b);pdf.set_text_color(255,255,255)
            pdf.set_font("Helvetica","B",20)
            pdf.set_xy(xp,yp);pdf.cell(bw,13,str(counts.get(cls,0)),align="C",fill=True)
            pdf.set_xy(xp,yp+13);pdf.set_font("Helvetica","",6.5)
            pdf.set_fill_color(max(0,r-30),max(0,g-30),max(0,b-30))
            pdf.cell(bw,4,cls,align="C",fill=True,ln=(1 if i==4 else 0))
        pdf.ln(9)

        # Tooth chart image
        pdf.set_font("Helvetica","B",10);pdf.set_text_color(30,58,95)
        pdf.cell(0,7,"FDI TOOTH CHART",ln=True)
        pdf.image(tmp.name,x=14,w=182)
        pdf.ln(4)

        # Priority findings
        if risk_teeth:
            pdf.set_font("Helvetica","B",10);pdf.set_text_color(185,28,28)
            pdf.cell(0,7,f"PRIORITY FINDINGS  —  {len(risk_teeth)} teeth require attention",ln=True)
            for t,v in risk_teeth[:12]:
                urgent=RISK[v["status"]]==3
                bg=(254,242,242)if urgent else(255,247,237)
                bd=(185,28,28)if urgent else(194,65,12)
                pdf.set_fill_color(*bg);pdf.set_draw_color(*bd);pdf.set_line_width(.3)
                pdf.rect(14,pdf.get_y(),182,7,"FD")
                pdf.set_xy(16,pdf.get_y()+1)
                pdf.set_font("Helvetica","B",8);pdf.set_text_color(*bd)
                pdf.cell(22,5,f"[{'URGENT'if urgent else'MONITOR'}]")
                pdf.set_font("Helvetica","",8);pdf.set_text_color(71,85,105)
                pdf.cell(24,5,f"Tooth {t}")
                pdf.cell(52,5,TOOTH_NAMES.get(t,""))
                pdf.set_text_color(*bd);pdf.cell(38,5,v["status"])
                pdf.set_text_color(100,116,139)
                pdf.cell(0,5,f"{v['conf']*100:.0f}% confidence",ln=True)
            pdf.ln(3)

        # ── PAGE 2: full table ────────────────────────────────
        pdf.add_page()
        pdf.set_fill_color(30,58,95);pdf.rect(0,0,210,18,"F")
        pdf.set_text_color(255,255,255);pdf.set_font("Helvetica","B",13)
        pdf.set_y(5);pdf.cell(0,9,"COMPLETE TOOTH-BY-TOOTH FINDINGS",ln=True,align="C")
        pdf.ln(7)

        hdrs  =["FDI","Tooth Name","Jaw","Condition","Confidence","Risk"]
        col_ws=[14,52,18,38,26,22]
        pdf.set_font("Helvetica","B",8)
        for h,w in zip(hdrs,col_ws):
            pdf.set_fill_color(30,58,95);pdf.set_text_color(255,255,255)
            pdf.cell(w,8,h,border=1,fill=True,align="C")
        pdf.ln();alt=False
        for t,v in sorted(results.items()):
            if pdf.get_y()>268:pdf.add_page()
            pdf.set_fill_color(248,250,252)if alt else pdf.set_fill_color(241,245,249)
            alt=not alt
            cr,cg,cb=BOX_RGB[v["status"]]
            rr,rg,rb=RISK_RGB[RISK[v["status"]]]
            row=[str(t),TOOTH_NAMES.get(t,""),
                 "Upper"if t in FDI_UPPER else"Lower",
                 v["status"],f"{v['conf']*100:.0f}%",RISK_LABEL[RISK[v["status"]]]]
            for i,(cell,w)in enumerate(zip(row,col_ws)):
                if   i==3:pdf.set_text_color(cr,cg,cb);pdf.set_font("Helvetica","B",8)
                elif i==5:pdf.set_text_color(rr,rg,rb);pdf.set_font("Helvetica","",8)
                else:     pdf.set_text_color(51,65,85);pdf.set_font("Helvetica","",8)
                pdf.cell(w,6,cell,border=1,fill=True,align="C")
            pdf.ln()

        pdf.set_y(-18);pdf.set_text_color(148,163,184);pdf.set_font("Helvetica","",7)
        pdf.cell(0,5,"⚠  AI-generated. Always consult a licensed dental professional.",align="C")

        return bytes(pdf.output())
    finally:
        # Always clean up temp file even if PDF fails
        try: os.unlink(tmp.name)
        except: pass

# ════════════════════════════════════════════════════════════
# SIDEBAR
# ════════════════════════════════════════════════════════════
predict_fn, model_ready = load_model()

with st.sidebar:
    st.markdown("""
    <div style="padding:16px 0 6px;text-align:center">
      <div style="font-size:36px">🦷</div>
      <div style="font-size:16px;font-weight:700;color:#38bdf8;letter-spacing:.08em;margin-top:4px">DENTAL·AI</div>
      <div style="font-size:10px;color:#334155;margin-top:2px">EfficientNet-B3 · FDI Notation</div>
    </div>""",unsafe_allow_html=True)
    st.divider()
    if model_ready: st.success("✅ Trained model loaded")
    else:           st.info("🔄 Demo mode — add model to Drive for real results")
    st.divider()
    mode=st.radio("**Mode**",["🦷 Full OPG (32 teeth)","🔬 Single Tooth"])
    st.divider()
    st.markdown("**Patient Info** *(for PDF)*")
    p_name=st.text_input("Name",    placeholder="e.g. John Doe")
    p_age =st.text_input("Age",     placeholder="e.g. 35")
    p_date=st.text_input("Date",    value=datetime.date.today().strftime("%d %b %Y"))
    st.divider()
    st.markdown("**Legend**")
    for cls in CLASS_NAMES:
        st.markdown(f"<div style='display:flex;align-items:center;gap:8px;padding:3px 0'>"
                    f"<span style='color:{COLOR[cls]};font-size:15px'>{ICON[cls]}</span>"
                    f"<span style='color:#94a3b8;font-size:12px'>{cls}</span></div>",
                    unsafe_allow_html=True)

# ════════════════════════════════════════════════════════════
# HEADER
# ════════════════════════════════════════════════════════════
st.markdown("""
<div style="background:linear-gradient(135deg,#1e3a5f,#1d4ed8);
            border-radius:18px;padding:24px 32px;margin-bottom:20px">
  <div style="font-size:23px;font-weight:800;color:white">🦷 Dental AI Diagnostic System</div>
  <div style="font-size:12px;color:#93c5fd;margin-top:4px">
    AI-Assisted Radiograph Analysis · EfficientNet-B3 · FDI International Notation</div>
</div>""",unsafe_allow_html=True)

if not model_ready:
    st.warning("Demo mode — results are simulated. Add best_dental_model.pth to Drive.",icon="🔄")

uploaded=st.file_uploader("📁 Upload Dental X-ray (JPG / PNG)",type=["jpg","jpeg","png"])

if uploaded is None:
    st.markdown("""
    <div style="background:white;border:1.5px solid #e2e8f0;border-radius:18px;
                padding:56px 0;text-align:center;margin-top:14px">
      <div style="font-size:54px;margin-bottom:10px">🦷</div>
      <div style="font-size:18px;font-weight:600;color:#1e3a5f;margin-bottom:6px">
        Upload a dental X-ray to begin</div>
      <div style="font-size:13px;color:#94a3b8">Periapical · Bitewing · Panoramic OPG</div>
    </div>""",unsafe_allow_html=True)
    st.stop()

raw    =uploaded.read()
img_rgb=decode_img(raw)
if img_rgb is None:
    st.error("❌ Cannot decode image.");st.stop()

# ════════════════════════════════════════════════════════════
# SINGLE TOOTH
# ════════════════════════════════════════════════════════════
if "Single" in mode:
    col1,col2=st.columns([1.3,1],gap="large")
    with col1:
        st.image(img_rgb,caption=f"📷 {uploaded.name}",use_container_width=True)
    with col2:
        with st.spinner("Analyzing…"):
            pred,conf,scores=predict_fn(img_rgb)
        c=COLOR[pred];risk=RISK[pred]
        risk_lbl=["","🟢 Low","🟠 Medium","🔴 High"][risk]
        st.markdown(f"""
        <div style="background:linear-gradient(135deg,#1e3a5f,#1d4ed8);
                    border-radius:16px;padding:22px;color:white;margin-bottom:14px">
          <div style="font-size:10px;letter-spacing:.2em;opacity:.65;margin-bottom:6px">AI DIAGNOSIS</div>
          <div style="font-size:30px;font-weight:800">{ICON[pred]}&nbsp;{pred}</div>
          <div style="display:flex;gap:24px;font-size:13px;opacity:.85;margin-top:10px">
            <span>Confidence: <b>{conf*100:.1f}%</b></span>
            <span>Risk: <b>{risk_lbl if risk else "🟢 None"}</b></span>
          </div>
          <div style="background:rgba(255,255,255,.2);border-radius:6px;height:8px;overflow:hidden;margin-top:12px">
            <div style="background:white;width:{int(conf*100)}%;height:100%;border-radius:6px"></div>
          </div>
        </div>""",unsafe_allow_html=True)
        st.markdown('<div style="background:#f8fafc;border:1px solid #e2e8f0;border-radius:14px;padding:16px">',unsafe_allow_html=True)
        st.markdown('<div style="font-size:11px;font-weight:700;color:#64748b;letter-spacing:.1em;margin-bottom:10px">ALL CLASS PROBABILITIES</div>',unsafe_allow_html=True)
        for cls,sc in sorted(scores.items(),key=lambda x:-x[1]):
            cc=COLOR[cls]
            st.markdown(f"""
            <div style="margin-bottom:9px">
              <div style="display:flex;justify-content:space-between;font-size:12px;margin-bottom:3px">
                <span style="color:{cc};font-weight:600">{ICON[cls]}&nbsp;{cls}</span>
                <span style="color:#475569;font-weight:700">{sc*100:.1f}%</span>
              </div>
              <div style="background:#e2e8f0;border-radius:4px;height:7px;overflow:hidden">
                <div style="background:{cc};width:{int(sc*100)}%;height:100%;border-radius:4px"></div>
              </div>
            </div>""",unsafe_allow_html=True)
        st.markdown("</div>",unsafe_allow_html=True)

# ════════════════════════════════════════════════════════════
# FULL OPG — all downloads guaranteed here
# ════════════════════════════════════════════════════════════
else:
    st.image(img_rgb,
             caption=f"{uploaded.name}  ·  {img_rgb.shape[1]}×{img_rgb.shape[0]} px",
             use_container_width=True)
    st.divider()

    with st.spinner("🔍 Analyzing all 32 teeth…"):
        results=run_opg(predict_fn,raw)
    if results is None:
        st.error("Could not process image.");st.stop()

    st.success(f"✅ Analysis complete — {len(results)} teeth processed")

    # Metrics
    counts=Counter(v["status"]for v in results.values())
    for col,cls in zip(st.columns(5),CLASS_NAMES):
        col.metric(f"{ICON[cls]} {cls}",counts.get(cls,0))

    # Risk panel
    risk_teeth=sorted([(t,v)for t,v in results.items()if RISK[v["status"]]>=2],
                      key=lambda x:-RISK[x[1]["status"]])
    if risk_teeth:
        st.divider()
        st.markdown("#### ⚠️ Priority Findings")
        cols=st.columns(min(len(risk_teeth),3))
        for i,(t,v) in enumerate(risk_teeth[:6]):
            c=COLOR[v["status"]];urgent=RISK[v["status"]]==3
            with cols[i%3]:
                st.markdown(f"""
                <div style="background:{'#fef2f2'if urgent else'#fff7ed'};
                            border:1.5px solid {c};border-radius:14px;padding:14px 16px;margin-bottom:8px">
                  <div style="font-size:10px;font-weight:700;color:{c};letter-spacing:.1em">
                    {'🔴 URGENT'if urgent else'🟠 MONITOR'}</div>
                  <div style="font-size:22px;font-weight:800;color:{c};margin:4px 0">Tooth {t}</div>
                  <div style="font-size:11px;color:#64748b">{TOOTH_NAMES.get(t,'')}</div>
                  <div style="font-size:12px;color:{c};margin-top:4px;font-weight:600">
                    {ICON[v['status']]}&nbsp;{v['status']}</div>
                  <div style="font-size:10px;color:#94a3b8">{v['conf']*100:.0f}% confidence</div>
                </div>""",unsafe_allow_html=True)

    st.divider()
    st.markdown("#### 🦷 FDI Color-Coded Tooth Chart")

    # Generate chart — if it fails, show error but still offer CSV
    chart_png = None
    try:
        with st.spinner("Rendering chart…"):
            chart_png = make_chart(results)
        st.image(chart_png, use_container_width=True)
    except Exception as e:
        st.warning(f"Chart render issue: {e}")

    st.divider()
    st.markdown("#### 📋 Detailed Results Table")

    df=pd.DataFrame([{
        "FDI":       t,
        "Name":      TOOTH_NAMES.get(t,""),
        "Jaw":       "Upper"if t in FDI_UPPER else"Lower",
        "Condition": v["status"],
        "Conf %":    round(v["conf"]*100,1),
        "Risk":      RISK_LABEL[RISK[v["status"]]],
        "_r":        RISK[v["status"]],
    }for t,v in results.items()])

    fa,fb=st.columns(2)
    with fa:
        filt=st.multiselect("Filter:",CLASS_NAMES,default=CLASS_NAMES)
    with fb:
        srt=st.selectbox("Sort:",["Tooth Number","Confidence ↓","Risk ↓"])

    df2=df[df["Condition"].isin(filt)].copy()
    if srt=="Confidence ↓":  df2=df2.sort_values("Conf %",ascending=False)
    elif srt=="Risk ↓":      df2=df2.sort_values("_r",ascending=False)
    else:                    df2=df2.sort_values("FDI")

    st.dataframe(
        df2[["FDI","Name","Jaw","Condition","Conf %","Risk"]]
           .style.applymap(lambda v:f"color:{COLOR.get(v,'#334155')}",subset=["Condition"]),
        use_container_width=True,
        height=min(60+len(df2)*36,500))

    # ══════════════════════════════════════════════════════
    # DOWNLOAD SECTION  ← always rendered, independent
    # ══════════════════════════════════════════════════════
    st.divider()
    st.markdown("#### ⬇️ Download Reports")
    st.markdown(
        "<p style='color:#64748b;font-size:13px;margin-bottom:14px'>"
        "All three formats are ready to download instantly.</p>",
        unsafe_allow_html=True)

    dl1,dl2,dl3=st.columns(3)

    # 1) CSV — always works
    with dl1:
        csv_bytes=make_csv(results).encode("utf-8")
        st.download_button(
            label="📄 Download CSV",
            data=csv_bytes,
            file_name=f"dental_report_{datetime.date.today()}.csv",
            mime="text/csv",
            use_container_width=True,
            key="dl_csv")

    # 2) PNG chart
    with dl2:
        if chart_png:
            st.download_button(
                label="🖼️ Download Chart PNG",
                data=chart_png,
                file_name=f"tooth_chart_{datetime.date.today()}.png",
                mime="image/png",
                use_container_width=True,
                key="dl_png")
        else:
            st.info("Chart unavailable")

    # 3) PDF — wrapped in try/except with user feedback
    with dl3:
        if chart_png:
            try:
                pdf_bytes=make_pdf(results,chart_png,p_name,p_age,p_date)
                st.download_button(
                    label="📑 Download PDF Report",
                    data=pdf_bytes,
                    file_name=f"dental_report_{datetime.date.today()}.pdf",
                    mime="application/pdf",
                    use_container_width=True,
                    key="dl_pdf")
            except Exception as e:
                st.error(f"PDF error: {e}")
                # Fallback: offer CSV if PDF fails
                st.download_button(
                    label="📄 Fallback CSV",
                    data=csv_bytes,
                    file_name=f"dental_report_{datetime.date.today()}.csv",
                    mime="text/csv",
                    use_container_width=True,
                    key="dl_csv_fallback")
        else:
            st.info("Needs chart first")

    st.markdown("""
    <div style="text-align:center;padding:14px 0 2px;color:#94a3b8;font-size:11px">
      ⚠️ AI-generated results are for informational purposes only.
      Always consult a licensed dental professional.
    </div>""",unsafe_allow_html=True)
'''

with open("dental_dashboard.py","w") as f:
    f.write(dashboard_code)
print("✅ dental_dashboard.py written")

import subprocess, time, re, urllib.request, sys

subprocess.run(["pkill","-f","streamlit"],capture_output=True)
time.sleep(2)

subprocess.Popen([
    "streamlit","run","dental_dashboard.py",
    "--server.port","8501",
    "--server.headless","true",
    "--server.enableCORS","false",
    "--server.enableXsrfProtection","false",
    "--server.maxUploadSize","30",
    "--runner.fastReruns","true",
],stdout=subprocess.DEVNULL,stderr=subprocess.DEVNULL)

print("⏳ Waiting for Streamlit…"); time.sleep(6)

try:
    colab_ip=urllib.request.urlopen("https://ipv4.icanhazip.com",timeout=5).read().decode().strip()
except:
    colab_ip="visit ipv4.icanhazip.com"

url=None
for attempt in range(4):
    lt=subprocess.Popen(["lt","--port","8501"],stdout=subprocess.PIPE,stderr=subprocess.PIPE)
    time.sleep(3)
    out=lt.stdout.readline().decode().strip()
    m=re.search(r"https://[^\s]+",out)
    if m: url=m.group(); break
    lt.kill(); time.sleep(2)

print("\n"+"="*56)
print("  ✅  DENTAL AI IS LIVE!")
print(f"  🌐  URL      : {url or 'run cell again — lt failed'}")
print(f"  🔑  Password : {colab_ip}")
print("="*56)
print("\n  After uploading an X-ray in OPG mode:")
print("  📄 Download CSV  |  🖼️ Download Chart PNG  |  📑 Download PDF")

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸
changed 22 packages in 1s
⠸
⠸3 packages are looking for funding
⠸  run `npm fund` for details
⠸✅ Packages ready
✅ dental_dashboard.py written
⏳ Waiting for Streamlit…

  ✅  DENTAL AI IS LIVE!
  🌐  URL      : https://seven-mangos-watch.loca.lt
  🔑  Password : 136.110.62.122

  After uploading an X-ray in OPG mode:
  📄 Download CSV  |  🖼️ Download Chart PNG  |  📑 Download PDF
